### Imports

In [1]:
import anndata as ad
import mudata as md
import numpy as np
import pandas as pd
import alphapepttools as at
from scipy.sparse import csr_matrix
from itertools import combinations
from scipy import sparse

/Users/mcthielert/miniforge3/envs/msmudata_dev/lib/python3.14/site-packages/alphaquant/quant_reader/longformat_reader.py:12: UserWarning: Dependency 'dask' not installed. If you want to use its functionality, install it with: `pip install "alphaquant[dask]"` . Falling back to non-dask based processing.
  warnings.warn(
/Users/mcthielert/miniforge3/envs/msmudata_dev/lib/python3.14/site-packages/alphaquant/ptm/ptmsite_mapping.py:28: UserWarning: Dependency 'dask' not installed. If you want to use its functionality, install it with: `pip install "alphaquant[dask]"` . ImportError will be raised if the data requires out-of-memory processing.
  warnings.warn(


## Functions

In [2]:
def get_unique_mappings(psm_path: str, feature_level_names: list[str]) -> pd.DataFrame:
    """
    Get unique mappings from PSM table for specified feature levels.
    
    Parameters:
    - psm_table: DataFrame containing PSM data with columns for each feature level.
    - feature_level_names: List of column names corresponding to feature levels (e.g., ["Precursor", "Protein"]).
    
    Returns:
    - DataFrame with unique mappings between the specified feature levels.
    """
    # Select only the relevant columns for mapping
    psm_df = pd.read_csv(psm_path, sep="\t")
    mapping_df = psm_df.loc[:, feature_level_names].drop_duplicates()
    
    return mapping_df

In [3]:
def sparse_matrix_mapping(unique_mapping_df: pd.DataFrame) -> csr_matrix:
    """
    Create a square sparse adjacency matrix for varp from a feature-level mapping.

    Parameters
    ----------
    mapping_df : pd.DataFrame
        DataFrame where the index contains source features (e.g., precursors)
        and each column contains target features (e.g., protein groups).
        Produced by get_unique_mappings().

    Returns
    -------
    csr_matrix
        Square adjacency matrix of shape (n_total, n_total) where
        n_total = n_source + n_target features.
    """
    all_values = pd.unique(unique_mapping_df.values.ravel())                                                                                   
    value_to_idx = {v: i for i, v in enumerate(all_values)}                                                                     
    n = len(all_values)                                                                                                         
                                                                                                                                
    rows, cols = [], []                                                                                                         
    for _, row in unique_mapping_df.iterrows():                                                                                                
        for v1, v2 in combinations(row.values, 2):                                                                                     
            i, j = value_to_idx[v1], value_to_idx[v2]                                                                           
            rows.extend([i, j])                                                                                                 
            cols.extend([j, i])                                                                                                 
                                                                                                                                
    adj = sparse.coo_matrix(                                                                                                    
        (np.ones(len(rows), dtype=np.float64), (rows, cols)),
        shape=(n, n),                                                                                                           
    ).tocsr()                                                                                                                     
    adj.data = np.ones_like(adj.data)  # collapse summed duplicates to 1 
    adj = pd.DataFrame.sparse.from_spmatrix(adj, index=all_values, columns=all_values)                                                                                                                      
    return adj


In [4]:
def create_mudata(psm_path: str, search_engine: str, feature_level_names: dict[str, str]) -> md.MuData:
    """
    create  a MuData object from the PSM table, including unique mappings between feature levels.
   
   Parameters:
    - psm_path: Path to the Diann PSM table file (e.g., TSV or CSV).
    - feature_level_names: List of column names in the PSM table that represent the feature levels to map (e.g., ["Precursor.Id", "Protein.Group"]).
    Returns:
    - DataFrame with unique mappings between the specified feature levels.
    """
    #Create mapping between feature levels
    mapping_df = get_unique_mappings(psm_path, feature_level_names.values())
    matrix_varp = sparse_matrix_mapping(mapping_df)

    # load all level adatas with alphapepttools functions
    adatas = {}
    for level in feature_level_names.keys():
        adatas[level] = at.io.read_psm_table(
            psm_path,
            level=level,
            search_engine=search_engine,
            feature_id_column=feature_level_names[level],
        )

    mudata = md.MuData(adatas)
    mudata.varp["feature_mapping"] = matrix_varp.reindex(index=mudata.var_names, columns=mudata.var_names)

    return mudata

### Result Path

In [5]:
prec_report_path = "../data/diann_1.8.1_report_head.tsv"

## Code

In [6]:
#report paths
prec_report_path_diann = "../data/diann_1.8.1_report_head.tsv"
#prec_report_path_alphadia = "../data/alphadia_2.0.0_psm_table.parquet"

In [7]:
features = {"precursor": "Precursor.Id", "proteins": "Protein.Group"}
msmudata_gen = create_mudata(prec_report_path_diann, search_engine="diann", feature_level_names=features)

  self._update_attr("var", axis=0, join_common=join_common)

  self._update_attr("obs", axis=1, join_common=join_common)



In [8]:
msmudata_gen

MuData object with n_obs × n_vars = 6 × 30
  varp:	'feature_mapping'
  2 modalities
    precursor:	6 x 16
    proteins:	6 x 14

In [9]:
#features = {"precursor": "Precursor.Id", "proteins": "Protein.Group"}
#msmudata_gen = create_mudata(prec_report_path_alphadia, search_engine="alphadia", feature_level_names=features)